# Mistral-7B-Instruct-v0.1 — Behavioral Baseline and Probing

**Phase 1 + Phase 2:** Behavioral baseline, rotation tests, and activation probing.

**Hardware:** A100 80GB | **Runtime:** ~45 minutes

**Step 1:** Run Cell 1, then Runtime → Restart session  
**Step 2:** Run all remaining cells top to bottom


In [1]:
# Cell 1 — Install (run once, then restart runtime)
!pip install -q numpy==1.26.4
!pip install -q transformer_lens scikit-learn datasets pandas matplotlib
print('Done. Restart runtime now: Runtime → Restart session')


Done. Restart runtime now: Runtime → Restart session


In [2]:
# Cell 2 — Config and imports (run after restart)
DRIVE_BASE = '/content/drive/MyDrive/identity-under-the-hood'  # change if needed

import os, re, torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from datasets import load_dataset
from transformer_lens import HookedTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    pass

os.makedirs(f'{DRIVE_BASE}/activations', exist_ok=True)
DEVICE = 'cuda'
INSTRUCT_LAYERS = [6, 12, 18, 24, 28, 31]
N_BEHAVIORAL = 100
N_ROTATION = 50
N_EXTRACT = 100
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Drive: {DRIVE_BASE}')
print('Ready.')


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Drive: /content/drive/MyDrive/identity-under-the-hood
Ready.


In [3]:
# Cell 3 — Load BBQ datasets
race_ds   = load_dataset('Elfsong/BBQ', split='race_ethnicity').to_pandas()
gender_ds = load_dataset('Elfsong/BBQ', split='gender_identity').to_pandas()

race_ambig   = race_ds[race_ds['context_condition'] == 'ambig'].reset_index(drop=True)
gender_ambig = gender_ds[gender_ds['context_condition'] == 'ambig'].reset_index(drop=True)

print(f'Race ambiguous: {len(race_ambig)}')
print(f'Gender ambiguous: {len(gender_ambig)}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/age-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

data/disability_status-00000-of-00001.pa(…):   0%|          | 0.00/85.2k [00:00<?, ?B/s]

data/gender_identity-00000-of-00001.parq(…):   0%|          | 0.00/217k [00:00<?, ?B/s]

...

Generating race_x_ses split:   0%|          | 0/11160 [00:00<?, ? examples/s]

Generating religion split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating ses split:   0%|          | 0/6864 [00:00<?, ? examples/s]

Generating sexual_orientation split:   0%|          | 0/864 [00:00<?, ? examples/s]

Race ambiguous: 3440
Gender ambiguous: 2836


In [4]:
# Cell 4 — Helper functions

def format_prompt(row):
    return (f"Context: {row['context']}\n"
            f"Question: {row['question']}\n"
            f"A) {row['ans0']}\nB) {row['ans1']}\nC) {row['ans2']}\nAnswer:")

def get_answer_logits(prompt, model):
    tokens = model.to_tokens(prompt)
    with torch.no_grad():
        logits = model(tokens)[0, -1, :]
    return {'A': logits[model.to_single_token(' A')].item(),
            'B': logits[model.to_single_token(' B')].item(),
            'C': logits[model.to_single_token(' C')].item()}

def probe_cv(X, y, seed=42):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    scores = []
    for tr, te in cv.split(X, y):
        scaler = StandardScaler()
        clf = LogisticRegression(max_iter=1000, random_state=seed)
        clf.fit(scaler.fit_transform(X[tr]), y[tr])
        scores.append(clf.score(scaler.transform(X[te]), y[te]))
    return np.mean(scores), np.std(scores)

def extract_at_pos(prompt, model, pos, layers):
    tokens = model.to_tokens(prompt)
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)
    return {l: cache[f'blocks.{l}.hook_resid_post'][0, pos, :].detach().cpu().float().numpy()
            for l in layers}

def find_token_pos(prompt, model, target):
    """Find token position. Handles split tokens (e.g. His+panic for Hispanic)."""
    tokens = model.to_tokens(prompt)
    strs = [model.to_string(tokens[0][i]) for i in range(tokens.shape[1])]
    # Exact match first
    for i, t in enumerate(strs):
        if t.strip() == target:
            return i
    # Split token: check consecutive pairs
    for i in range(len(strs) - 1):
        combined = strs[i].strip() + strs[i+1].strip()
        if combined == target:
            return i
    return None

print('Helpers defined.')


Helpers defined.


In [5]:
# Cell 5 — Load Mistral-7B-Instruct-v0.1
print('Loading Mistral-7B-Instruct-v0.1...')
model = HookedTransformer.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.1',
    device=DEVICE, dtype=torch.float16)
model.eval()
print('Loaded.')


Loading Mistral-7B-Instruct-v0.1...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Loaded pretrained model mistralai/Mistral-7B-Instruct-v0.1 into HookedTransformer
Loaded.


In [6]:
# Cell 6 — Phase 1: Behavioral baseline
categories = {
    'race_ethnicity': race_ambig.head(N_BEHAVIORAL),
    'gender_identity': gender_ambig.head(N_BEHAVIORAL),
    'disability_status': load_dataset('Elfsong/BBQ', split='disability_status').to_pandas()
                         .query('context_condition=="ambig"').head(N_BEHAVIORAL),
    'nationality': load_dataset('Elfsong/BBQ', split='nationality').to_pandas()
                   .query('context_condition=="ambig"').head(N_BEHAVIORAL),
}

print('Phase 1: Behavioral collapse test')
print(f'Sampling {N_BEHAVIORAL} ambiguous prompts across {len(categories)} categories...')

behavioral_results = {}
for cat_name, df in categories.items():
    counts = {'A': 0, 'B': 0, 'C': 0}
    for _, row in df.iterrows():
        try:
            logits = get_answer_logits(format_prompt(row), model)
            counts[max(logits, key=logits.get)] += 1
        except Exception:
            continue
    total = sum(counts.values())
    pcts = {k: round(100*v/total, 1) for k, v in counts.items()}
    behavioral_results[cat_name] = pcts
    print(f'{cat_name}: A={pcts["A"]}%, B={pcts["B"]}%, C={pcts["C"]}%')


Phase 1: Behavioral collapse test
Sampling 100 ambiguous prompts across 4 categories...
race_ethnicity: A=31.0%, B=33.0%, C=36.0%
gender_identity: A=31.0%, B=45.0%, C=24.0%
disability_status: A=24.0%, B=46.0%, C=30.0%
nationality: A=36.0%, B=28.0%, C=36.0%


In [7]:
# Cell 7 — Rotation tests
print('Rotation tests (50 prompts, not-answerable moved to position A)...')
print()

unknown_markers = ['cannot be determined', "can't be determined", 'unknown',
                    'not enough info', 'not answerable', 'not known',
                    'undetermined', "can't answer"]

rotation_results = {}
for cat_name, df in categories.items():
    selections = []
    for _, row in df.head(N_ROTATION * 3).iterrows():
        if len(selections) >= N_ROTATION:
            break
        # Find unknown answer using answer_info
        info = row.get('answer_info', {})
        unknown_key = None
        for key in ['ans0', 'ans1', 'ans2']:
            if any(m in str(info.get(key, [])).lower() for m in unknown_markers):
                unknown_key = key
                break
        if unknown_key is None:
            continue
        ans_list = [row['ans0'], row['ans1'], row['ans2']]
        unknown_ans = row[unknown_key]
        others = [a for k, a in zip(['ans0','ans1','ans2'], ans_list) if k != unknown_key]
        rotated = pd.Series({'context': row['context'], 'question': row['question'],
                             'ans0': unknown_ans, 'ans1': others[0], 'ans2': others[1]})
        try:
            logits = get_answer_logits(format_prompt(rotated), model)
            selections.append(max(logits, key=logits.get))
        except Exception:
            continue
    pct_a = round(100 * selections.count('A') / len(selections), 1) if selections else None
    rotation_results[cat_name] = pct_a
    print(f'{cat_name}: {pct_a}% selected A when correct answer placed in position A')

print()
print('Interpretation: ~33% = content sensitive. ~0% or ~100% = position bias.')


Rotation tests (50 prompts, not-answerable moved to position A)...

race_ethnicity: 80.0% selected A when correct answer placed in position A
gender_identity: 42.0% selected A when correct answer placed in position A
disability_status: 18.0% selected A when correct answer placed in position A
nationality: 10.0% selected A when correct answer placed in position A

Interpretation: ~33% = content sensitive. ~0% or ~100% = position bias.


In [8]:
# Cell 8 — Phase 2: Race/ethnicity activation extraction
# Hispanic tokenizes as 'His'+'panic' in Mistral — handled below

print('Extracting race/ethnicity activations...')

hispanic_records, black_records = [], []

for _, row in race_ambig.iterrows():
    ctx = row['context']
    prompt = format_prompt(row)

    if 'Hispanic' in ctx and len(hispanic_records) < N_EXTRACT:
        pos = find_token_pos(prompt, model, 'Hispanic')
        if pos is not None:
            acts = extract_at_pos(prompt, model, pos, INSTRUCT_LAYERS)
            hispanic_records.append({'demo_group': 'Hispanic',
                **{f'layer_{l}': acts[l] for l in INSTRUCT_LAYERS}})

    if 'Black' in ctx and 'African American' not in ctx and len(black_records) < N_EXTRACT:
        pos = find_token_pos(prompt, model, 'Black')
        if pos is not None:
            acts = extract_at_pos(prompt, model, pos, INSTRUCT_LAYERS)
            black_records.append({'demo_group': 'Black',
                **{f'layer_{l}': acts[l] for l in INSTRUCT_LAYERS}})

    if len(hispanic_records) >= N_EXTRACT and len(black_records) >= N_EXTRACT:
        break

n = min(len(hispanic_records), len(black_records))
all_race_records = hispanic_records[:n] + black_records[:n]
print(f'Hispanic: {len(hispanic_records)}, Black: {len(black_records)}, Balanced: {len(all_race_records)}')


Extracting race/ethnicity activations...
Hispanic: 100, Black: 100, Balanced: 200


In [9]:
# Cell 9 — Gender identity extraction
print('Extracting gender identity activations (man vs woman)...')

man_records, woman_records = [], []

for _, row in gender_ambig.iterrows():
    ctx = row['context']
    prompt = format_prompt(row)

    for label, records_list in [('man', man_records), ('woman', woman_records)]:
        if len(records_list) >= 24:
            continue
        if f' {label} ' not in ctx.lower() and not ctx.lower().startswith(label):
            continue
        pos = find_token_pos(prompt, model, label)
        if pos is None:
            continue
        acts = extract_at_pos(prompt, model, pos, INSTRUCT_LAYERS)
        records_list.append({'demo_group': label,
            **{f'layer_{l}': acts[l] for l in INSTRUCT_LAYERS}})

    if len(man_records) >= 24 and len(woman_records) >= 24:
        break

n = min(len(man_records), len(woman_records))
all_gender_records = man_records[:n] + woman_records[:n]
print(f'man: {len(man_records)}, woman: {len(woman_records)}, Balanced: {len(all_gender_records)}')


Extracting gender identity activations (man vs woman)...
man: 24, woman: 24, Balanced: 48


In [10]:
# Cell 10 — Linear probing
print('Phase 2: Linear probing')
print('5-fold CV logistic regression')
print('Chance baseline: 50.0%')
print()

probe_results = []

for category, records in [('race_ethnicity', all_race_records),
                           ('gender_identity', all_gender_records)]:
    print(f'{category.replace("_"," ").title()} (n={len(records)}):')
    for layer in INSTRUCT_LAYERS:
        X = np.array([r[f'layer_{layer}'] for r in records])
        y = np.array([r['demo_group'] for r in records])
        mean, std = probe_cv(X, y)
        print(f'  Layer {layer}: {mean:.1%} \u00b1 {std:.1%}')
        probe_results.append({'category': category, 'layer': layer,
                              'accuracy': round(mean, 4), 'std': round(std, 4)})
    print()


Phase 2: Linear probing
5-fold CV logistic regression
Chance baseline: 50.0%

Race Ethnicity (n=200):
  Layer 6: 100.0% ± 0.0%
  Layer 12: 100.0% ± 0.0%
  Layer 18: 100.0% ± 0.0%
  Layer 24: 100.0% ± 0.0%
  Layer 28: 100.0% ± 0.0%
  Layer 31: 100.0% ± 0.0%

Gender Identity (n=48):
  Layer 6: 100.0% ± 0.0%
  Layer 12: 100.0% ± 0.0%
  Layer 18: 100.0% ± 0.0%
  Layer 24: 100.0% ± 0.0%
  Layer 28: 100.0% ± 0.0%
  Layer 31: 100.0% ± 0.0%



In [11]:
# Cell 11 — Permutation validation
print('Phase 3: Permutation validation (100 iterations)')
print()

perm_results = []
N_PERM = 100

for category, records in [('race_ethnicity', all_race_records),
                           ('gender_identity', all_gender_records)]:
    layer = INSTRUCT_LAYERS[2]  # layer 18 — mid-network
    X = np.array([r[f'layer_{layer}'] for r in records])
    y = np.array([r['demo_group'] for r in records])
    real_mean, _ = probe_cv(X, y)
    perm_accs = []
    for _ in range(N_PERM):
        y_perm = np.random.permutation(y)
        acc, _ = probe_cv(X, y_perm)
        perm_accs.append(acc)
    perm_mean = round(np.mean(perm_accs) * 100, 1)
    perm_results.append({'category': category, 'layer': layer,
                         'permuted_accuracy': perm_mean})
    print(f'{category} (layer {layer}): permuted = {perm_mean}% \u00b1 {round(np.std(perm_accs)*100,1)}%')

print()
print('Expected if genuine signal: ~50% (chance)')


Phase 3: Permutation validation (100 iterations)

race_ethnicity (layer 18): permuted = 50.6% ± 5.0%
gender_identity (layer 18): permuted = 54.6% ± 9.3%

Expected if genuine signal: ~50% (chance)


In [2]:
# Cell 12 — Save and print summary
import json as json_module

results = {
    'model': 'Mistral-7B-Instruct-v0.1',
    'behavioral': behavioral_results,
    'rotation': rotation_results,
    'probe': probe_results,
    'permutation': perm_results,
}

save_path = f'{DRIVE_BASE}/activations/mistral_instruct_results.json'
with open(save_path, 'w') as f:
    json_module.dump(results, f, indent=2)
print(f'Saved: {save_path}')

import pandas as pd
df = pd.DataFrame(probe_results)
csv_path = f'{DRIVE_BASE}/activations/mistral_instruct_probe_results.csv'
df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

print()
print('='*65)
print('SUMMARY \u2014 Mistral-7B-Instruct-v0.1')
print('='*65)
print('Behavioral collapse:')
for cat, pcts in behavioral_results.items():
    print(f'  {cat}: A={pcts["A"]}%, B={pcts["B"]}%, C={pcts["C"]}%')
print('Rotation sensitivity:')
for cat, pct in rotation_results.items():
    print(f'  {cat}: {pct}%')
print('Probe accuracy (peak layers):')
for cat in ['race_ethnicity', 'gender_identity']:
    peak = max([r for r in probe_results if r['category']==cat], key=lambda r: r['accuracy'])
    print(f'  {cat}: {peak["accuracy"]:.1%} \u00b1 {peak["std"]:.1%} (layer {peak["layer"]})')
print()
print('Expected:')
print('  race_ethnicity: A=31%, B=33%, C=36%, rotation=32%, probe=100%')
print('  gender_identity: rotation=38%, probe=100%')


NameError: name 'behavioral_results' is not defined